In [6]:
from pyspark.sql import SparkSession
import os
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Inicializar Spark con soporte para PostgreSQL JDBC
def get_spark_session():
    jar_path = os.path.abspath("postgresql-42.7.5.jar")  # Ruta del driver JDBC

    return SparkSession.builder \
        .appName("Business Intelligence") \
        .config("spark.jars", jar_path) \
        .config("spark.driver.extraClassPath", jar_path) \
        .getOrCreate()

# Obtener conexión PostgreSQL desde Spark
def get_postgres_connection():
    url = "jdbc:postgresql://localhost:5432/postgres"
    properties = {
        "user": "postgres",
        "password": "postgres",
        "driver": "org.postgresql.Driver"
    }
    return url, properties

# Ejecutar consultas SQL en PostgreSQL usando Spark JDBC
def execute_query(spark, query):
    url, properties = get_postgres_connection()
    return spark.read.jdbc(url=url, table=f"({query}) as temp", properties=properties)

print("Generando reportes de Business Intelligence basados en el Data Warehouse.")

# Consultas SQL y visualización de datos
queries = {
    "Top 10 Productos más vendidos": """(
        SELECT p.nombre, SUM(hv.cantidad) AS total_vendido
        FROM hechos_ventas hv
        JOIN productos p ON hv.id_producto = p.id
        GROUP BY p.nombre, hv.id_producto
        ORDER BY total_vendido DESC
        LIMIT 10
    )""",
    
    "Top 5 Clientes con más pedidos": """(
        SELECT c.nombre, COUNT(hv.id_cliente) AS num_pedidos
        FROM hechos_ventas hv
        JOIN clientes c ON hv.id_cliente = c.id
        GROUP BY c.nombre, hv.id_cliente
        ORDER BY num_pedidos DESC
        LIMIT 5
    )""",
    
    "Top 5 Corresponsales con más pedidos": """(
        SELECT c.nombre, COUNT(*) AS num_pedidos
        FROM hechos_ventas hv
        JOIN dim_corresponsales c ON hv.id_corresponsal = c.id_corresponsal
        GROUP BY c.nombre, hv.id_corresponsal
        ORDER BY num_pedidos DESC
        LIMIT 5
    )""",
}

print("Consultas ejecutadas, mostrando resultados.")

# Inicializar Spark
spark = get_spark_session()

# Ejecutar las consultas con Spark JDBC
dataframes = {}
for titulo, query in queries.items():
    df = execute_query(spark, query)
    dataframes[titulo] = df
    #print(f"Resultado para '{titulo}':")
    #df.show()


pagina_actual = 1  # Página inicial
registros_por_pagina = 10  # Número de registros por página

tabla_output = widgets.Output()
grafico_output = widgets.Output()
botones_output = widgets.Output()
dropdown_reportes = widgets.Dropdown(options=list(dataframes.keys()), description="Reporte:")
boton_anterior = widgets.Button(description="⏪ Anterior")
boton_siguiente = widgets.Button(description="⏩ Siguiente")



def mostrar_grafico():
    reporte_seleccionado = dropdown_reportes.value
    df = dataframes[reporte_seleccionado]

    grafico_output.clear_output()
    with grafico_output:
        plt.figure(figsize=(8, 4))
        #sns.barplot(x=df.iloc[:, 0], y=df.iloc[:, 1], palette="Blues_r")
        sns.barplot(x=df.iloc[:, 0], y=df.iloc[:, 1], hue=df.iloc[:, 0], palette="Blues_r", legend=False)
        plt.title(reporte_seleccionado)
        plt.xlabel(df.columns[0])
        plt.ylabel(df.columns[1])
        plt.xticks(rotation=45)
        plt.show()



def mostrar_pagina(event=None):
    global pagina_actual
    reporte_seleccionado = dropdown_reportes.value
    df = dataframes[reporte_seleccionado]

    if event is not None:
        pagina_actual = 1
    
    total_paginas = (df.count() // registros_por_pagina) + (1 if df.count() % registros_por_pagina > 0 else 0)

    inicio = (pagina_actual - 1) * registros_por_pagina
    fin = inicio + registros_por_pagina

    df_paginated = df.limit(fin).filter(df.index >= inicio)

    df_paginated.index = range(inicio + 1, inicio + 1 + len(df_paginated))

    # Limpiar y mostrar nueva salida
    tabla_output.clear_output()
    with tabla_output:
        print(f"\n📌 {reporte_seleccionado} - Página {pagina_actual} de {total_paginas}\n")
        display(df_paginated)

    
    

    
    botones_output.clear_output()
    with botones_output:
        if len(df) > 10:
            display(widgets.HBox([boton_anterior, boton_siguiente]))

    
    grafico_output.clear_output()
    with grafico_output:
        if len(df) <= 10:
            mostrar_grafico()
            #display(grafico_output)




def actualizar_pagina(event):
    global pagina_actual
    df = dataframes[dropdown_reportes.value]
    total_paginas = (df.count() // registros_por_pagina) + (1 if df.count() % registros_por_pagina > 0 else 0)

    if event.description == "⏪ Anterior" and pagina_actual > 1:
        pagina_actual -= 1
    elif event.description == "⏩ Siguiente" and pagina_actual < total_paginas:
        pagina_actual += 1

    mostrar_pagina()




# 📌 Asignar eventos
dropdown_reportes.observe(mostrar_pagina, names="value")
boton_anterior.on_click(actualizar_pagina)
boton_siguiente.on_click(actualizar_pagina)


# Inicializar la primera visualización
mostrar_pagina(pagina_actual)
display(dropdown_reportes)
display(tabla_output)
display(botones_output)
display(grafico_output)


Generando reportes de Business Intelligence basados en el Data Warehouse.
Consultas ejecutadas, mostrando resultados.


AttributeError: 'DataFrame' object has no attribute 'index'